# Bars from a tick stream: `Resample` and string aggregations

Raw market data arrives as a *tick stream*: one row per trade, each with a
price, a volume, and a timestamp. `Resample` condenses that stream into
fixed-interval **bars**.

This notebook covers the simplest way to drive it: the `freq=` bucket width
and the built-in **string aggregations**. We start with a single-column `mean`
bar, then build the standard OHLCV candle, then split volume into buyer- and
seller-initiated flow with `ohlcv2`.

Every bar is **causal**: its value depends only on ticks already seen inside
the bar, never on future ones. A bucket is emitted only once a later timestamp
proves it complete; the trailing partial bucket is flushed at end of input.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from screamer import Resample

rng = np.random.default_rng(42)
n = 80

# Irregular integer timestamps: each tick lands 1-2 units after the last.
timestamps = np.cumsum(rng.integers(1, 3, size=n)).astype(np.int64)

# Mid-price: a slow random walk around 100.
price = 100.0 + np.cumsum(rng.standard_normal(n) * 0.3)

# Trade size: heavy-tailed, always positive.
volume = rng.exponential(scale=50.0, size=n)

# Signed volume: + = buyer-initiated, - = seller-initiated (size drawn
# independently of sign, a deliberately simple microstructure model).
signed_volume = volume * rng.choice([-1.0, 1.0], size=n)

pd.DataFrame({
    "timestamp":     timestamps[:6],
    "price":         price[:6].round(3),
    "volume":        volume[:6].round(1),
    "signed_volume": signed_volume[:6].round(1),
}).set_index("timestamp")

## The simplest bar: one column, one reducer

Pass a 1-D value array, its `index` (the timestamps), and a bucket width
`freq=`. With a string `agg` like `"mean"`, each bar reduces to a single
number.

The return value is a `Stream`. It unpacks as `(values, index)` for
convenience, and single-column results expose `.values` and `.index` directly.
Other single-column shorthands work the same way: `first`, `last`, `min`,
`max`, `sum`, `count`.

In [ ]:
BAR = 20

bars = Resample(freq=BAR, agg="mean")(price, timestamps)

# Resample returns a (values, index) tuple:
values, index = bars
print("bar starts:", index)
print("bar means: ", values.round(3))


## OHLCV candles

For price bars, pass a two-column `[price, volume]` array and `agg="ohlcv"`.
Each bar yields five **positional** columns (`open`, `high`, `low`, `close`,
`volume`) collected causally from the ticks inside it.

The return value is a `(values, index)` tuple. Columns are positional in
documented order: `values[:, 0]` = open, `values[:, 1]` = high, `values[:, 2]` = low,
`values[:, 3]` = close, `values[:, 4]` = volume.


In [ ]:
_OHLCV_COLUMNS = ("open", "high", "low", "close", "volume")

ohlcv_vals, ohlcv_idx = Resample(freq=BAR, agg="ohlcv")(
    np.column_stack([price, volume]), timestamps)

print("columns (positional):", _OHLCV_COLUMNS)
pd.DataFrame(
    ohlcv_vals,
    index=pd.Index(ohlcv_idx, name="bar_open"),
    columns=list(_OHLCV_COLUMNS),
).round(3)


In [ ]:
# Candlestick view: OHLC range on top, bar volume below.
fig, (ax_price, ax_vol) = plt.subplots(
    2, 1, figsize=(9, 5), sharex=True,
    gridspec_kw={"height_ratios": [2, 1]},
)

for t, row in zip(ohlcv_idx, ohlcv_vals):
    o, h, l, c = row[:4]
    color = "steelblue" if c >= o else "crimson"
    ax_price.plot([t, t], [l, h], color="0.4", lw=1.5, zorder=1)   # high-low wick
    ax_price.bar(t, c - o, bottom=o, width=BAR * 0.5,              # open-close body
                 color=color, alpha=0.8, zorder=2)

ax_price.set_ylabel("price")
ax_price.set_title(f"OHLCV bars  (freq={BAR})")

ax_vol.bar(ohlcv_idx, ohlcv_vals[:, 4], width=BAR * 0.5, color="0.6")  # volume col 4
ax_vol.set_ylabel("volume")
ax_vol.set_xlabel("bar open index")
plt.tight_layout()


## Buyer- vs seller-initiated volume: `ohlcv2`

`agg="ohlcv2"` expects a `[price, signed_volume]` input and returns six
positional columns: open(0), high(1), low(2), close(3), buy_vol(4), sell_vol(5).


In [ ]:
_OHLCV2_COLUMNS = ("open", "high", "low", "close", "buy_vol", "sell_vol")

ohlcv2_vals, ohlcv2_idx = Resample(freq=BAR, agg="ohlcv2")(
    np.column_stack([price, signed_volume]), timestamps)

print("columns (positional):", _OHLCV2_COLUMNS)
net_flow = ohlcv2_vals[:, 4] - ohlcv2_vals[:, 5]  # buy_vol col4 - sell_vol col5

fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(ohlcv2_idx,  ohlcv2_vals[:, 4],  width=BAR * 0.5, color="seagreen", label="buy")
ax.bar(ohlcv2_idx, -ohlcv2_vals[:, 5], width=BAR * 0.5, color="crimson",  label="sell")
ax.plot(ohlcv2_idx, net_flow, color="0.2", lw=1.5, marker="o", ms=4, label="net flow")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("bar open index")
ax.set_ylabel("signed volume")
ax.set_title("Buyer- vs seller-initiated volume per bar")
ax.legend(fontsize=9)
plt.tight_layout()
